PROJECT 1 : LIVE MARKET DATA PIPELINE & DASHBOARD

In [ ]:
## installing the yfinance package that fetches real stock from yahoo finance

# !pip install yfinance pandas

In [3]:
## asking for price data of last five days from y finance


import yfinance as yf

""""
apple = yf.Ticker("AAPL")
data = apple.history(period ="5d")

data
"""

'"\napple = yf.Ticker("AAPL")\ndata = apple.history(period ="5d")\n\ndata\n'

In [ ]:
## Defining a list of stickers and looping through them to collect each ones data

"""
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN","TLSA"]
all_data = {}
for ticker in tickers:
    stock = yf.Ticker(ticker)
    all_data[ticker] = stock.history(period = "5d")
## We just automated fetching for different companies at once, instead of doing it manually five times   
all_data["AAPL"]
"""

In [7]:
## Storing as a csv file
import pandas as pd

"""all_data["AAPL"].to_csv("AAPL_prices.csv")"""

'all_data["AAPL"].to_csv("AAPL_prices.csv")'

In [ ]:
## using the loop to store the five different stickers as csv files

"""
for ticker in tickers:
    print(f"Saving{ticker}...")
    all_data[ticker].to_csv(f"{ticker}_prices.csv")
print("Done")

## This creates a csv file that does not append in case of a rerun, but rather overwrites, to solve this issue, we use databases (SQL)
"""

In [ ]:
## But python does not talk to SQL directly, so we need a translator package

"""!pip install sqlalchemy pyodbc"""

In [1]:
## creating a connection object btw python and sql
from sqlalchemy import create_engine

server = 'localhost'
database = 'finance_pipeline'

connection_string = f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
## driver is the specific software translator, trusted_connection tells it to use Window Authentication
engine = create_engine(connection_string)

In [10]:
## Testing if the connection went through
import pandas as pd
test = pd.read_sql("SELECT 1 AS test_column", engine)

In [ ]:
test
## since this printed, it shows that the connection went through and that python successfully talked to sql

In [12]:
## saving our real data into sql
aapl_df = all_data["AAPL"].reset_index()
aapl_df["Date"] = aapl_df["Date"].dt.tz_localize(None) ## this is done to scrap out the timezone, to enable sql read the data

aapl_df.to_sql("AAPL_prices", engine, if_exists="append", index=False)
## the append is there to cancel theproblem of overwriting, it adds the new data anytime we rerun the code#
## On the other hand, this will produce duplicates as every rerun appends new rows whether already existing or not
## for this cause, the pipeline needs the rule **only insert a row if it is not already there**

5

In [ ]:
## Fetching for all five stickers
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

for ticker in tickers:
    stock = yf.Ticker(ticker)
    month_data = stock.history(period="3mo").reset_index()
    month_data["Date"] = month_data["Date"].dt.tz_localize(None)

## Building the rule to eliminate duplicates
    existing = pd.read_sql(f"SELECT DISTINCT Date FROM {ticker}_prices", engine)
    existing_dates = existing["Date"].tolist()
    
    new_data = month_data[~month_data["Date"].isin(existing_dates)]

## Storing it in the databse
    if len(new_data) > 0:
        new_data.to_sql(f"{ticker}_prices", engine, if_exists="append", index=False)
        print(f"{ticker}: added {len(new_data)} rows")
    else:
        print(f"{ticker}: already up to date")

In [4]:
## In conclusion, the working funtion for updating sticker is;

def update_ticker(ticker_symbol, table_name):
    stock = yf.Ticker(ticker_symbol)
    new_fetch = stock.history(period="5d").reset_index()
    new_fetch["Date"] = new_fetch["Date"].dt.tz_localize(None)

    try:
        existing = pd.read_sql(f"SELECT DISTINCT Date FROM {table_name}", engine)
        existing_dates = existing["Date"].tolist()
    except:
        existing_dates = []

    new_data = new_fetch[~new_fetch["Date"].isin(existing_dates)]

    if len(new_data) > 0:
        new_data.to_sql(table_name, engine, if_exists="append", index=False)
        print(f"{ticker_symbol}: added {len(new_data)} new rows")
    else:
        print(f"{ticker_symbol}: no new data to add")

update_ticker("AAPL","AAPL_prices")

## History period of 5d is there incase we miss a day, so we can update the missed dates

AAPL: added 5 new rows


In [5]:
## running the function for all 5 stickers
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

for ticker in tickers:
    table_name = f"{ticker}_prices"
    update_ticker(ticker, table_name)

AAPL: added 5 new rows
MSFT: added 5 new rows
GOOGL: added 5 new rows
AMZN: added 5 new rows
TSLA: added 5 new rows


In [8]:
tables = pd.read_sql("SELECT table_name FROM information_schema.tables",engine)
print(tables)

     table_name
0   MSFT_prices
1  GOOGL_prices
2   AMZN_prices
3   TSLA_prices
4   AAPL_prices


In [19]:
## Next we will be visualizing our datasets on streamlit
## streamlit is a python tool that turns a plain script into an interactive webpage without needing any web development knowledge

In [9]:
## Checking for updates
check = pd.read_sql("SELECT TOP 5 * FROM AAPL_prices ORDER BY Date DESC",engine)
check

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2026-07-13,317.015015,323.450012,315.779999,316.850006,29424251,0.0,0.0
1,2026-07-13,317.015015,323.450012,315.779999,317.000000,29436614,0.0,0.0
2,2026-07-10,314.720001,316.390015,312.170013,315.730011,18560726,0.0,0.0
3,2026-07-10,314.720001,316.910004,312.170013,315.320007,34109200,0.0,0.0
4,2026-07-10,314.720001,316.910004,312.170013,315.320007,34109200,0.0,0.0
